In [2]:
import yfinance as yf
import pandas as pd

df = yf.download("PETR4.SA", start="2020-01-01", auto_adjust=True)
precos = df["Close"]


[*********************100%***********************]  1 of 1 completed


5.1 Preço, retorno simples e log-retorno

In [3]:
import numpy as np

ret_simples = precos.pct_change()
ret_log = np.log(precos / precos.shift(1))

5.2 Base 252 e anualização

In [4]:
ret_anual = (1 + ret_simples).prod() ** (252 / len(ret_simples))
vol_anual = ret_simples.std() * np.sqrt(252)

5.3 CDI: a taxa livre de risco brasileira

In [12]:
from bcb import sgs

cdi = sgs.get({"cdi": 12}, start="2017-01-01")
cdi = cdi["cdi"] / 100

5.4 Sharpe e Sortino

In [15]:
def sharpe(ret, cdi):
    excesso = ret - cdi.reindex(ret.index).fillna(0)
    return excesso.mean() / excesso.std() * np.sqrt(252)


def sortino(ret, cdi):
    excesso = ret - cdi.reindex(ret.index).fillna(0)
    downside = excesso[excesso < 0].std()
    return excesso.mean() / downside * np.sqrt(252)

5.5 Drawdown

In [16]:
def drawdown(ret):
    curva = (1 + ret).cumprod()
    pico = curva.cumax()
    return curva / pico - 1

def max_drawdown(ret):
    return drawdown(ret).min()

5.6 Correlação

In [17]:
retornos_varios.corr()

NameError: name 'retornos_varios' is not defined

5.7 O básico de séries temporais

In [18]:
from statsmodels.tsa.stattools import adfuller

print("p-valor preço:", adfuller(precos.dropna())[1])
print("p-valor retorno:", adfuller(ret_simples.dropna())[1])

p-valor preço: 0.9880796663106731
p-valor retorno: 3.3167369380654165e-24


/tmp/ipykernel_14151/1255585843.py:3: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  print("p-valor preço:", adfuller(precos.dropna())[1])
/tmp/ipykernel_14151/1255585843.py:4: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  print("p-valor retorno:", adfuller(ret_simples.dropna())[1])
